# Report Extraction

This notebook extracts the clinical variables from the echocardiography reports (`Report_ecoTT`) with a language model, following the meeting with the supervisor (20/10/2026). It starts from the de-identified reports saved in `Data_Preparation` Section 6.4., so the patient list is not needed here.

Before any report is sent to an external model, the private information still left in the text is found and removed.

## Table of contents

1. [Setup](#1-setup)
2. [Private information left in the reports](#2-private-information-left-in-the-reports)
3. [Age at the evaluation](#3-age-at-the-evaluation)
4. [Removing the private information](#4-removing-the-private-information)
5. [Testing language models on a sample of reports](#5-testing-language-models-on-a-sample-of-reports)
    - 5.1. [ChatGPT](#51-chatgpt)
    - 5.2. [Claude](#52-claude)
    - 5.3. [Gemini](#53-gemini)
    - 5.4. [MedGemma](#54-medgemma)
    - 5.5. [Comparison with the database](#55-comparison-with-the-database)
6. [Choosing the variables](#6-choosing-the-variables)

### 1. Setup
[[go back to the top]](#table-of-contents)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the project root importable, wherever the notebook is
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import re
import numpy as np
import pandas as pd
import json
import ollama


from src.multiscope import *

reports = pd.read_pickle(REPORTS_FILE)
print(f"De-identified reports: {len(reports)}")

De-identified reports: 250


### 2. Private information left in the reports
[[go back to the top]](#table-of-contents)

`Data_Preparation` removed the patient's name, the health number (`NSC:`) and the numbers with 6 or more digits. This section counts how many reports still contain other private information, such as, the birth date, the exam number, the exam date, the names of who performed and validated the exam, and any other date. Only counts are printed, never the text.

In [2]:
PRIVATE = {
    "birth date": r"Nascimento",
    "exam number": r"N[º°o] Exame",
    "exam date": r"Data\s*Exame|\bDat\b",
    "signature (Executado / Validado por)": r"Executado|Validado",
    "Dr. / Dra. / Téc.": r"\b(?:Dr|Dra|Téc)\.",
    "any date (dd/mm/yyyy or dd-mm-yyyy)": r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
}
pd.Series({field: reports.str.contains(pattern).sum() for field, pattern in PRIVATE.items()}, name="reports")

birth date                              238
exam number                             234
exam date                               235
signature (Executado / Validado por)    198
Dr. / Dra. / Téc.                       192
any date (dd/mm/yyyy or dd-mm-yyyy)     245
Name: reports, dtype: int64

To remove this information, its position in the report has to be known. The next cell counts how many reports have each of the parts that seem to delimit the clinical text, the indication for the exam at the start, and the place and date and the signature at the end.

In [3]:
PARTS = {
    "Indicação para exame (start of the clinical text)": r"Indica[çc][ãa]o para exame",
    "Relatório (title of the conclusions)": r"\nRelat[óo]rio\s*\n",
    "Vila Nova de Gaia (place and date)": r"Vila Nova de Gaia",
    "Executado por (signature)": r"Executado",
}
pd.Series({part: reports.str.contains(pattern).sum() for part, pattern in PARTS.items()}, name="reports")

Indicação para exame (start of the clinical text)    229
Relatório (title of the conclusions)                 139
Vila Nova de Gaia (place and date)                   203
Executado por (signature)                            198
Name: reports, dtype: int64

- **Header (birth date 238, exam number 234, exam date 235):** almost every report starts with these fields.

- **Signature (198) and `Dr.` / `Dra.` / `Téc.` (192):** the names of who performed and validated the exam are at the end, after `Executado por`.

- **Any date (245):** besides the birth date and the exam date, some reports also have dates in the clinical text (e.g. a previous exam or surgery).

The parts that delimit the clinical text are not in every report (`Indicação para exame` in 229 of 250, `Vila Nova de Gaia` in 203), because the reports do not all have the same layout. So instead of keeping only the text between two parts, the private information is removed by rules that do not depend on the layout (see [[Section 4.]](#4-removing-the-private-information)).


### 3. Age at the evaluation
[[go back to the top]](#table-of-contents)

The database has no age column, but most reports have the birth date (`Data de Nascimento`). Before the dates are removed, the age at the evaluation is computed from it and the `EvaluationDate` of the database (the exam date in the report is sometimes split across two lines, because of the layout in two columns).

`birth_date` (in `src/multiscope.py`) takes the earliest full date of the report, since every other date (the exam, previous exams) is later. Reports without the `Data de Nascimento` field get no age, because their earliest date is not a birth date. The age is kept in its own column, not in the text.

In [ ]:
db = load_db_version(next(DB_DIR.glob(CLINICAL_DB_GLOB)))
evaluation = pd.to_datetime(db.set_index("ID")["EvaluationDate"], dayfirst=True, errors="coerce")

birth = reports.map(birth_date)
AGE_CAP = 87   # ages of 87 or older are grouped, so that the oldest group has at least 5 patients
age = np.floor((evaluation.reindex(reports.index) - birth).dt.days / 365.25).clip(upper=AGE_CAP)

print(f"Reports with a birth date: {birth.notna().sum()} of {len(reports)}")
age.describe().round(1)

Reports with a birth date: 238 of 250


count    238.0
mean      68.1
std       12.9
min       22.0
25%       59.0
50%       68.5
75%       79.0
max       91.0
dtype: float64

238 of the 250 reports have a birth date, the same number of reports with the `Data de Nascimento` field in [[Section 2]](#2-private-information-left-in-the-reports), so every one of them gets an age. The other 12 reports have no age.

The ages go from 22 to 91 years (median 68.5), all plausible for the patients of the study, which shows that the earliest date of each report is indeed the birth date and not another date.

Very old ages are rare, so they can help identify a patient together with the other fields of the header (sex, height, weight). So the ages of 87 or older are stored as 87, the lowest cap that groups at least 5 patients (9 patients, 1 of them older than 90). The models lose at most 4 years of precision for these patients.

### 4. Removing the private information
[[go back to the top]](#table-of-contents)

`remove_private` (in `src/multiscope.py`):

1. cuts the report at `Executado`, which removes the signature block (who performed and validated the exam);
2. replaces the exam numbers (2 or 3 capital letters followed by 9 or more digits) with `[ID]`;
3. replaces every date (`dd/mm/yyyy`, `dd-mm-yy`, `dd.mm.yyyy`) and every year alone (`19xx` or `20xx`) with `[DATE]`.
4. replaces the number with 4 or 5 digits right after a `[NAME]` (the number of who signed the report, found in 1 report) with `[ID]`.
5. replaces the number with 5 or more digits glued to `Final` or `Preliminary` (the status of the report, followed by the number of who signed it, found in 8 reports) with `[ID]`. The rule of 6 or more digits in `Data_Preparation` does not catch it, because there is no space between the word and the number.

The labels (`Data de Nascimento:`, `Nº Exame:`) stay, only their values are replaced. The next cell applies it and counts how many reports still have each type of private information.

In [5]:
llm_reports = reports.map(remove_private)

LEFT = {
    "exam number": r"[A-Z]{2,3}\d{6,}",
    "Dr. / Dra. / Téc.": r"\b(?:Dr|Dra|Téc)\.",
    "date": r"(?<!\d)\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}",
    "year (19xx or 20xx)": r"(?<!\d)(?:19|20)\d{2}(?!\d)",
    "number after [NAME]": r"\[NAME\]\s*\d{4,5}\b",
    "number glued to a word": r"[A-Za-z]\d{5,}",
    "any number with 5+ digits": r"(?<![\d,.])\d{5,}(?![\d,.])",
}
print(pd.Series({field: llm_reports.str.contains(pattern).sum() for field, pattern in LEFT.items()}, name="reports still with it"), "\n")
print(f"Share of the text kept (median): {(llm_reports.str.len() / reports.str.len()).median():.0%}")

exam number                  0
Dr. / Dra. / Téc.            0
date                         0
year (19xx or 20xx)          0
number after [NAME]          0
number glued to a word       0
any number with 5+ digits    0
Name: reports still with it, dtype: int64 

Share of the text kept (median): 94%


In [6]:
pd.DataFrame({"age": age, "report": llm_reports}).rename_axis("ID").to_csv(LLM_REPORTS_FILE)
print(f"Saved {len(llm_reports)} reports to {LLM_REPORTS_FILE.name}")

Saved 250 reports to reports_for_llm.csv


After `remove_private`, no report has an exam number, a `Dr.` / `Dra.` / `Téc.`, a date, a year, or a number with 5 or more digits left (including the ones glued to a word, as the signer numbers after `Final` / `Preliminary` were). Since the names only appeared in the signature, cutting at `Executado` removed all of them. The median report keeps 94% of its text, so almost only the header values and the signature were removed, and the clinical content is intact.

Together with `Data_Preparation` Section 6.4. (patient name, `NSC:` and identifier numbers), the reports no longer have any direct identifier. What is left in the header (sex, height, weight, IMC) does not identify a patient on its own. 

The database is also saved in a de-identified version, with all its columns, so that it can be analysed outside this computer. The identifier columns were already removed by `load_db_version` (name, NSC, process number, birth date). Besides that, `EvaluationDate` and `EcoTT` are removed (dates that could help identify a patient), the age is added, `Report_ecoTT` is replaced by the de-identified report of this section, and `remove_private` is applied to every other column with text, in case it has dates or signatures.

In [7]:
shared = db.set_index("ID").drop(columns=["EvaluationDate", "EcoTT", REPORT_COLUMN])
text = shared.select_dtypes("object").columns
shared[text] = shared[text].apply(lambda s: s.map(lambda v: remove_private(v) if isinstance(v, str) else v))
shared = shared.join(pd.DataFrame({"age": age, "report": llm_reports}))

left = shared.astype(str).apply(lambda s: s.str.contains(r"\b(?:Dr|Dra|Téc)\.|(?<!\d)\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}|\b\d{4}-\d{2}-\d{2}\b|[A-Za-z]\d{5,}")).sum()
print("Columns still with a Dr./Téc. or a date:", left[left > 0].to_dict())

shared.to_csv(SHARED_DB_FILE)
print(f"Saved {len(shared)} patients and {shared.shape[1]} columns to {SHARED_DB_FILE.name}")

Columns still with a Dr./Téc. or a date: {}
Saved 301 patients and 37 columns to db_deidentified.csv


C:\Users\aleja\AppData\Local\Temp\ipykernel_23172\944229758.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text = shared.select_dtypes("object").columns


No column has a `Dr.` / `Téc.` or a date left. The de-identified database has the 301 patients of the database (including the 12 without signals), with 37 columns, the 35 of the database without `EvaluationDate`, `EcoTT` and `Report_ecoTT`, plus `age` and `report` (empty for the patients without a real report).

### 5. Testing language models on a sample of reports
[[go back to the top]](#table-of-contents)

Following the meeting with the supervisor (20/10/2026), before choosing the variables to extract from all the reports, four language models are tested on the same small sample of reports, to see which variables come out and if they make sense:

- **ChatGPT**, **Claude** and **Gemini**, three general models used through their external services. The reports sent are the ones saved in [[Section 4.]](#4-removing-the-private-information), without any private information.
- **MedGemma**, an open model trained on medical data, run locally on this computer, so the reports never leave it.

The four models receive the same reports and the same instructions. Their answers are checked against the report itself and against the columns of the database that describe the same findings (`IM`, `IT`, `IAO`, `IP`, `FEVEBP`, `FEVE4C`, `Ritmo`).

To be able to check every answer, the sample has 10 reports chosen at random only among the patients with all these columns filled in the database, with a fixed seed so that it is always the same. It is saved in `data/processed/reports_sample.csv`, the file given to the four models.

---

**How the test is done**

- **Prompt:** the four models receive exactly the same instructions, written in the cell below (`PROMPT`). As this is a first exploration, the prompt does not list the variables that are expected. It asks for every clinical variable written in each report, only what is written (nothing inferred), with the name, the value, the unit and the piece of text where it was found.

- **Answer format:** each model answers in JSON, with one entry per report (its `ID` and the list of variables). A fixed format makes it possible to read the four answers here and compare them automatically, instead of by eye.

- **Where the answers are saved:** each answer is saved as a file in `data/processed/` (`llm_chatgpt.json`, `llm_gemini.json`, `llm_claude.json` and `llm_medgemma.json`), not in this notebook. The answers contain clinical information from the reports, and the notebook is pushed to GitHub, while `data/` stays on this computer. The notebook only shows counts and comparisons.

- **ChatGPT, Gemini and Claude:** used in the browser, each one in a new chat, with the prompt and `reports_sample.csv` attached. A new chat starts without any previous context, so no model knows which variables are expected, and the test is fair between them. Before sending, the option to use the chats to train the models and the memory of previous chats are turned off in the settings of the three. The answer is downloaded as a `.json` file.

- **MedGemma:** run locally on this computer, with the same prompt and the same reports. Its answer is saved directly in `llm_medgemma.json` by the code.

---

**The prompt:**

```
The attached CSV file has echocardiography reports in European Portuguese (column "report"), one per patient (column "ID"). Private information was replaced by [NAME], [ID] and [DATE].

For each report, extract every clinical variable that is written in it. Do not infer or calculate anything, only what is written. Give each variable a short name in English (snake_case), its value and unit as written, and the piece of text where you found it.

Answer only with a JSON array, in this format:

[
  {"ID": 64, "variables": [
    {"name": "lv_ejection_fraction", "value": "30", "unit": "%", "evidence": "fração de ejeção de 30%"}
  ]}
]

Give the answer as a downloadable .json file, not as text in the chat.
```


In [2]:
PROMPT = """The attached CSV file has echocardiography reports in European Portuguese (column "report"), one per patient (column "ID"). Private information was replaced by [NAME], [ID] and [DATE].

For each report, extract every clinical variable that is written in it. Do not infer or calculate anything, only what is written. Give each variable a short name in English (snake_case), its value and unit as written, and the piece of text where you found it.

Answer only with a JSON array, in this format:

[
  {"ID": 64, "variables": [
    {"name": "lv_ejection_fraction", "value": "30", "unit": "%", "evidence": "fração de ejeção de 30%"}
  ]}
]

Give the answer as a downloadable .json file, not as text in the chat."""

In [ ]:
CHECK_COLUMNS = ["IM", "IT", "IAO", "IP", "FEVEBP", "FEVE4C", "Ritmo"]   # DB columns with the same findings as the reports

llm = pd.read_csv(LLM_REPORTS_FILE, index_col="ID")
check = db.set_index("ID").loc[llm.index, CHECK_COLUMNS]
complete = (check.notna() & ~check.astype(str).apply(lambda s: s.str.strip()).isin(["", "NA", "nan"])).all(axis=1)

if LLM_SAMPLE_FILE.exists():   # the same 10 patients given to the models, with the latest version of their reports
    sample = llm.loc[pd.read_csv(LLM_SAMPLE_FILE, index_col="ID").index]
else:
    sample = llm[complete].sample(10, random_state=42).sort_index()
sample.to_csv(LLM_SAMPLE_FILE)
print(f"Reports with all the columns filled: {complete.sum()} of {len(llm)}")
print(f"Sample saved to {LLM_SAMPLE_FILE.name}:", sample.index.tolist())

Reports with all the columns filled: 163 of 250
Sample saved to reports_sample.csv: [64, 138, 145, 221, 228, 244, 261, 266, 280, 291]


163 of the 250 reports have all the columns filled in the database, and the 10 reports of the sample are chosen among them.

When the sample was given to ChatGPT, Claude and Gemini, report 280 still had the number of who signed it (5 digits after `[NAME]`, not caught by the rule of 6 or more digits). It is not patient data, and it was removed afterwards from `reports_for_llm.csv` and from this sample, which was saved again with the same 10 reports. The answers of the models are not affected, as none of the variables uses that number.

The same check later found 8 more reports with the number of who signed them, glued to `Final` or `Preliminary` (see [[Section 4.]](#4-removing-the-private-information)). None of them is in the sample, so the test was not affected.

**What is a variable**

Each model answers, for each report, with a list of items. Each item is one piece of clinical information found in the report, here called a **variable**, with four parts:

- `name`: what it is, a short name in English chosen by the model;
- `value`: the value as written in the report;
- `unit`: its unit, when there is one;
- `evidence`: the piece of the report where the model found it, to check the value.

For example, in report 64, the ejection fraction written in the conclusion ("fração de ejeção estimada em 53% (pelo método de Simpson biplano)") was extracted by Claude as:

    {"name": "lv_ejection_fraction_report", "value": "53", "unit": "%",
     "evidence": "fração de ejeção estimada em 53% (pelo método de Simpson biplano)"}

The prompt does not give the names, so each model chooses its own, and the same variable can have different names in different reports or models. For the variables to be put in a table (one column per variable, one row per patient), the same finding must always have the same name. That is why `summary` shows, besides the number of variables per report, how consistent the names are:

- **different names:** how many different names the model used;
- **in every report:** how many names appear in all the reports, the higher the better;
- **in only one report:** how many names were used only once, the lower the better;
- **same name twice in a report:** how many variables repeat a name in the same report, which makes it unclear which value is which.

More variables is not necessarily better, the count also includes simple fields (height, weight), and the same finding extracted twice (from a table and from the conclusion). Whether the values are correct is checked separately, against the database.

In [9]:
def summary(answers):
    """Variables per report, and how consistent the names are across the reports."""
    reports_per_name = answers.groupby("name")["ID"].nunique()
    print("Variables per report:", answers.groupby("ID").size().to_dict())
    print(f"Total: {len(answers)} | different names: {len(reports_per_name)} | "
          f"in every report: {(reports_per_name == answers['ID'].nunique()).sum()} | in only one report: {(reports_per_name == 1).sum()} | "
          f"same name twice in a report: {answers.duplicated(['ID', 'name']).sum()}")

#### 5.1. ChatGPT
[[go back to the topic]](#5-testing-language-models-on-a-sample-of-reports)

The sample of reports and the instructions are given to ChatGPT, and its answer is saved, to see which variables it extracts from each report.

In [12]:
chatgpt = load_answers(OUTPUT_DIR / "llm_chatgpt.json")
summary(chatgpt)

Variables per report: {64: 45, 138: 45, 145: 41, 221: 39, 228: 39, 244: 11, 261: 45, 266: 61, 280: 55, 291: 41}
Total: 422 | different names: 113 | in every report: 9 | in only one report: 59 | same name twice in a report: 0


ChatGPT extracted 422 variables, between 11 (report 244) and 61 (report 266), from the tables and from the written conclusion. The names are not consistent across the reports, of 113 different names, only 9 are in every report and 59 in only one. The same finding sometimes gets two names (`rhythm` and `cardiac_rhythm`, `mitral_regurgitation` and `mitral_regurgitation_grade`), and the tables and the conclusion give separate variables for the same finding.

A first answer, given as text in the chat, was not valid JSON (two wrong escapes). It was asked again as a downloadable file, which is the one used here.

#### 5.2. Claude
[[go back to the topic]](#5-testing-language-models-on-a-sample-of-reports)

The same sample of reports and the same instructions are given to Claude, and its answer is saved, to compare it with the other models.

In [13]:
claude = load_answers(OUTPUT_DIR / "llm_claude.json")
summary(claude)

Variables per report: {64: 55, 138: 49, 145: 52, 221: 43, 228: 43, 244: 11, 261: 53, 266: 79, 280: 62, 291: 44}
Total: 491 | different names: 123 | in every report: 9 | in only one report: 59 | same name twice in a report: 0


Claude extracted more variables, 491, between 11 (report 244) and 79 (report 266), from the tables and from the written conclusion. Of 123 different names, 9 are in every report and 59 in only one. It separates the tables from the conclusion almost always with the same rule (`mitral_regurgitation` from the table and `mitral_regurgitation_report` from the conclusion), and splits combined values (e.g. `V.M. 4/1` into the maximum and the mean gradient). Some values keep the final full stop (`Normal.`, `Não.`).

#### 5.3. Gemini
[[go back to the topic]](#5-testing-language-models-on-a-sample-of-reports)

The same sample of reports and the same instructions are given to Gemini (3.1 Pro), and its answer is saved, to compare it with the other models.

In [14]:
gemini = load_answers(OUTPUT_DIR / "llm_gemini.json")
summary(gemini)

Variables per report: {64: 31, 138: 31, 145: 30, 221: 25, 228: 26, 244: 7, 261: 32, 266: 47, 280: 33, 291: 24}
Total: 286 | different names: 54 | in every report: 6 | in only one report: 19 | same name twice in a report: 4


Gemini extracted 286 variables, between 7 (report 244) and 47 (report 266). Its names are the most consistent, 54 different names, 19 in only one report. But from the written conclusion it takes only the numbers (e.g. the ejection fraction), and leaves out the qualitative findings (dilatation, hypertrophy, systolic function, wall motion) and some fields of the header, such as the rhythm (`Ritmo`). In report 266 the same name is used for different valves (three `ero` and three `regurgitant_volume`), so 4 variables have a repeated name in the same report.

#### 5.4. MedGemma
[[go back to the topic]](#5-testing-language-models-on-a-sample-of-reports)

MedGemma is run locally with Ollama, so the reports never leave this computer. Only the 4B version is used, the 27B needs about 17 GB, more than the 6 GB of the GPU (NVIDIA RTX 4050 Laptop) and the 16 GB of memory of this computer. The other three models are much larger, so part of any difference may come from the size and not only from the medical training.

The instructions are the same `PROMPT`, with two differences in how they are sent:

- **One report at a time:** a 4B model does not handle the 10 reports in a single answer well (the answer is very long and tends to be cut). Each report is sent separately, and the 10 answers are joined in `llm_medgemma.json`.
- **Fixed JSON format:** Ollama receives the format of the answer (`SCHEMA`), so the model can only answer with valid JSON in that format. The temperature is 0, to keep the answers as stable as possible.

In [4]:
sample = pd.read_csv(LLM_SAMPLE_FILE, index_col="ID")   # the same 10 reports given to the other models

In [ ]:
MEDGEMMA_FILE = OUTPUT_DIR / "llm_medgemma.json"
SCHEMA = {   # the answer for one report: its ID and the list of variables
    "type": "object",
    "properties": {
        "ID": {"type": "integer"},
        "variables": {"type": "array", "items": {
            "type": "object",
            "properties": {"name": {"type": "string"}, "value": {"type": "string"},
                           "unit": {"type": "string"}, "evidence": {"type": "string"}},
            "required": ["name", "value", "unit", "evidence"]}},
    },
    "required": ["ID", "variables"],
}

if MEDGEMMA_FILE.exists():   # keep the answers described below, the model does not give exactly the same answers in every run
    print(f"{MEDGEMMA_FILE.name} already exists, delete it to run MedGemma again")
else:
    answers, failed = [], []
    for pid, row in sample.iterrows():
        message = f"{PROMPT}\n\nHere there is only one report, given as text instead of a file. Answer only with the JSON object of this report.\n\nID: {pid}\nReport:\n{row['report']}"
        response = ollama.chat(model="medgemma:4b", messages=[{"role": "user", "content": message}],
                               format=SCHEMA, options={"temperature": 0, "num_ctx": 8192, "repeat_penalty": 1.2, "num_predict": 4096})
        try:
            answers.append(json.loads(response["message"]["content"]))
            print(f"ID {pid}: {len(answers[-1]['variables'])} variables")
        except json.JSONDecodeError:   # answer cut, the model got stuck repeating text
            failed.append(pid)
            print(f"ID {pid}: failed (answer cut)")

    MEDGEMMA_FILE.write_text(json.dumps(answers, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {len(answers)} answers to {MEDGEMMA_FILE.name} | failed: {failed}")

ID 64: 10 variables
ID 138: 14 variables
ID 145: 14 variables
ID 221: failed (answer cut)
ID 228: 25 variables
ID 244: 8 variables
ID 261: 43 variables
ID 266: 11 variables
ID 280: 49 variables
ID 291: 31 variables
Saved 9 answers to llm_medgemma.json | failed: [221]


In a first run, MedGemma got stuck in a loop in the first report (the same 9 variables repeated 13 times, until the answer was cut). A repetition penalty (`repeat_penalty` of 1.2) was added to avoid it.

MedGemma 4B answered 9 of the 10 reports, with very different numbers of variables, from 8 (report 244, which is short and also has few variables in the other models) to 49 (report 280). In several reports it extracts much less than the other models, for example only 11 variables in report 266, the most complex of the sample, against 47 to 79 in the other models. In report 221 the answer was cut, because the model got stuck repeating text inside a field, so that report has no answer.

The temperature is 0, but in a previous run report 145 gave 17 variables instead of 14, so on this GPU the answers are not exactly the same from one run to the next.

In [15]:
medgemma = load_answers(MEDGEMMA_FILE)
summary(medgemma)

Variables per report: {64: 10, 138: 14, 145: 14, 228: 25, 244: 8, 261: 43, 266: 11, 280: 49, 291: 31}
Total: 205 | different names: 154 | in every report: 0 | in only one report: 131 | same name twice in a report: 12


Its names are the least consistent, 154 different names for 205 variables, none in every report and 131 in only one, and 12 variables have a repeated name in the same report.

#### 5.5. Comparison with the database
[[go back to the topic]](#5-testing-language-models-on-a-sample-of-reports)

The database has columns with some of the findings of the reports, the regurgitation of each valve (`IAO`, `IM`, `IT`, `IP`), the rhythm (`Ritmo`) and the ejection fraction (`FEVEBP`, `FEVE4C`). The values extracted by each model are compared with them.

As each model names the variables differently, the values are found by the `evidence`, not by the name, for each column, the first variable whose evidence has the label of the report (`V.Ao`, `V.M`, `V.T`, `V.P`, `Ritmo`, `F.E. (BP)`, `F.E. (4C)`) and a value that can be compared.

The database stores codes, not text, and the codes are given in the database file itself, 0 to 5 for the regurgitation (`não`, `mínima`, `grau I`, `grau II`, `grau III`, `grau IV`, written in the reports as `Não`, `Mínima`, `Grau I/IV`... `Grau IV/IV`) and 0 to 3 for the rhythm (`Sinusal`, `FA`, `PACE`, `FLA`). The ejection fractions are compared with a tolerance of 1 point, because the database has decimals and the reports are rounded.

The table will consist in this format:

- Each row is a model: ChatGPT, Claude, Gemini, MedGemma.
- Each column is a DB variable: 
    - IAO, IM, IT, IP: regurgitation of the aortic, mitral, tricuspid and pulmonary valves;
    - Ritmo: the rhythm;
    - FEVEBP and FEVE4C: the ejection fraction.

Each cell is “a / b”, counted over the 10 reports of the sample:
- b is how many reports the model gave a value for that variable in;
- a is how many of those values are the same as the DB.

In [22]:
CHECKS = {"IAO": "V.Ao", "IM": "V.M", "IT": "V.T", "IP": "V.P", "Ritmo": "Ritmo", "FEVEBP": "F.E. (BP)", "FEVE4C": "F.E. (4C)"}   # column: label in the report
CODES = {"não": 0, "mínima": 1, "grau i/iv": 2, "grau ii/iv": 3, "grau iii/iv": 4, "grau iv/iv": 5,   # regurgitation, as coded in the database
         "sinusal": 0, "fibrilação auricular": 1, "pace": 2, "flutter auricular": 3}                   # rhythm, as coded in the database
NUMERIC = ["FEVEBP", "FEVE4C"]                                                                         # compared as numbers


def to_code(value, column):
    """Database code of an extracted value (a number for the ejection fractions)."""
    text = str(value).strip().rstrip(".").lower()
    return pd.to_numeric(text.replace(",", "."), errors="coerce") if column in NUMERIC else CODES.get(text)


def extracted(answers):
    """For each report and column, the first value whose evidence has the label of that column."""
    columns = {}
    for column, label in CHECKS.items():
        found = answers[answers["evidence"].astype(str).str.contains(label, regex=False)]
        found = found.assign(code=found["value"].map(lambda v: to_code(v, column))).dropna(subset=["code"])
        columns[column] = found.groupby("ID")["code"].first()
    return pd.DataFrame(columns).reindex(index=sample.index, columns=list(CHECKS)).astype(float)


truth = db.set_index("ID").loc[sample.index, list(CHECKS)].apply(pd.to_numeric, errors="coerce")
tolerance = pd.Series({"FEVEBP": 1, "FEVE4C": 1}).reindex(truth.columns, fill_value=0)

models = {"ChatGPT": chatgpt, "Claude": claude, "Gemini": gemini, "MedGemma": medgemma}
agree = {name: (extracted(a) - truth).abs().le(tolerance).sum() for name, a in models.items()}
found = {name: extracted(a).notna().sum() for name, a in models.items()}
pd.DataFrame(agree).T.astype(str) + " / " + pd.DataFrame(found).T.astype(str)

,IAO,IM,IT,IP,Ritmo,FEVEBP,FEVE4C
ChatGPT,9 / 9,9 / 9,9 / 9,8 / 8,7 / 8,9 / 9,0 / 0
Claude,9 / 9,9 / 9,9 / 9,8 / 8,7 / 8,9 / 9,0 / 0
Gemini,9 / 9,9 / 9,9 / 9,8 / 8,0 / 0,9 / 9,0 / 0
MedGemma,0 / 0,0 / 0,0 / 0,0 / 0,3 / 3,5 / 5,0 / 0


Each cell shows how many values agree with the database, of the values found by the label of the report (out of the 10 reports):

- **ChatGPT, Claude and Gemini** agree with the database in every regurgitation they found (9 of 9 for `IAO`, `IM` and `IT`, 8 of 8 for `IP`) and in every `FEVEBP` (9 of 9). The missing values are not in the reports, report 244 has no Doppler table (but the database has its regurgitations), the line of the pulmonary valve in report 291 has no grade, and report 228 only has `F.E. (est.)`, not `F.E. (BP)`.

- **Ritmo:** ChatGPT and Claude found it in 8 reports (reports 266 and 291 have no `Ritmo` field) and agree in 7. In the other one (report 221), the report says `Sinusal` and the database `1` (`FA`), so the difference is between the report and the database, not an error of the model. Gemini did not extract the rhythm.

- **FEVE4C** is not written in any of the 10 reports, so it cannot be checked. It may come from another source (e.g. the echocardiography machine).

- **MedGemma** found almost nothing this way (3 rhythms and 5 ejection fractions, all correct), because it did not follow the instructions, it translated some values to English (`minimal`, `mild`), wrote others in its own way (`I/IV`), and its evidence does not have the labels of the table. Some of its values are right when read by hand, but they cannot be compared automatically.

So, for the variables that can be checked, the three general models extract the values correctly, and the main difference between them is which variables they extract and how they name them.

### 6. Choosing the variables
[[go back to the top]](#table-of-contents)

The test of [[Section 5.]](#5-testing-language-models-on-a-sample-of-reports) showed that, with an open prompt, the models extract the values correctly but name them differently. The final prompt will give a fixed list of variables, chosen by three criteria.

**1. The goals of the thesis.** Following the thesis proposal, the report supervises each signal with the findings it can reflect:

- **PCG:** acoustically relevant valvular findings (regurgitation and stenosis of each valve, prostheses, pulmonary pressure);
- **ECG:** rhythm and cardiac remodelling (hypertrophy, dilatation of the chambers, wall motion);
- **PCG + ECG:** the overall echocardiographic phenotype (systolic function of both ventricles, ejection fraction).

The variables must also cover the three phenotypes used to evaluate the representations, such as, pulmonary hypertension probability, clinically significant valvular regurgitation and reduced left ventricular ejection fraction.

**2. How many reports have each one.** A variable is only useful if it is written in most reports.

**3. What the database already has.** Some findings are already in the database as columns. For those, extracting them from the report only fills the gaps and checks the values. The others can only come from the reports.

The next cell counts, for each candidate, how many of the 250 reports mention it, by searching for its label or its usual words in the text (only counts are printed).

The search is by words, so some counts mix chambers (e.g. `função sistólica` is written for both ventricles, `dilatação` for any chamber). They show whether a finding is usually reported, not how many reports describe the left ventricle.

In [ ]:
CANDIDATES = {   # group: {candidate variable: words or label in the report}
    "PCG": {
        "regurgitation per valve (Doppler table)": r"Tipo\s*V[áa]lvula",
        "regurgitation in the conclusion": r"insufici[êe]ncia (?:mitral|a[óo]rtica|tric[úu]spide|pulmonar)",
        "stenosis": r"estenose",
        "transvalvular gradients": r"gradiente|Grad m[áa]x",
        "valve area": r"[áa]rea (?:funcional|valvular)|[ÁA]rea \(cm",
        "prosthesis": r"pr[óo]t(?:\.|ese)",
        "PSAP (pulmonary pressure)": r"PSAP",
        "HTP probability": r"probabilidade",
        "pulmonary flow": r"Fluxo Pulmonar",
    },
    "ECG": {
        "rhythm": r"Ritmo:",
        "heart rate": r"Freq\.\s*Card",
        "septum thickness": r"Septo IV",
        "posterior wall thickness": r"Parede Posterior",
        "LV mass index": r"[ÍI]ndice de Massa",
        "hypertrophy": r"hipertrofia",
        "LA diameter": r"Diam AE",
        "LA volume": r"Vol AE|iVol AE",
        "LV diameter": r"VE d,",
        "dilatation of any chamber (conclusion)": r"dilata[çc][ãa]o",
        "wall motion": r"hipocin|acin[ée]sia|discin|contra[c]?tilidade segmentar",
    },
    "PCG + ECG": {
        "ejection fraction (BP, table)": r"F\.E\.\s*\(BP\)",
        "ejection fraction (est., table)": r"F\.E\.\s*\(est",
        "ejection fraction (conclusion)": r"fra[çc][ãa]o de eje[çc][ãa]o|\bFE\b",
        "systolic function, LV or RV": r"fun[çc][ãa]o (?:ventricular )?sist[óo]lica",
        "TAPSE": r"TAPSE|TASE",
        "S' tricuspid": r"S\W{0,2}\s*tric[úu]spide",
        "pericardial effusion": r"derrame peric[áa]rd",
        "inferior vena cava": r"veia cava",
    },
}

coverage = pd.DataFrame([
    {"group": group, "variable": name, "reports": llm_reports.str.contains(pattern, case=False).sum()}
    for group, variables in CANDIDATES.items() for name, pattern in variables.items()
])
coverage

,group,variable,reports
0,PCG,regurgitation per valve (Doppler table),199
1,PCG,regurgitation in the conclusion,203
2,PCG,stenosis,28
3,PCG,transvalvular gradients,143
4,PCG,valve area,48
5,PCG,prosthesis,55
6,PCG,PSAP (pulmonary pressure),122
7,PCG,pulmonary flow,199
8,ECG,rhythm,172
9,ECG,heart rate,52


The next cell counts, for the columns of the database with the same findings, how many of the 250 patients with a report have a value.

In [8]:
DB_COLUMNS = ["Ritmo", "IAO", "IM", "IT", "IP", "Alt_Valv_Aortica", "Alt Valv Mitral", "Alt Valv Pulmonar", "Alt Valv Tricúspide",
              "protese_aortica", "FEVEBP", "FEVE4C", "Fluxo pulmonar", "PSAP", "Gradiente VD/AD", "VMaxRegTric", "HTP Probability",
              "mPAP", "pCWP", "RVP"]

in_db = db.set_index("ID").loc[llm_reports.index, DB_COLUMNS]
filled = (in_db.notna() & ~in_db.astype(str).apply(lambda s: s.str.strip()).isin(["", "NA", "nan"])).sum()
filled.to_frame("patients with a value")

,patients with a value
Ritmo,244
IAO,243
IM,242
IT,243
IP,208
Alt_Valv_Aortica,240
Alt Valv Mitral,240
Alt Valv Pulmonar,219
Alt Valv Tricúspide,221
protese_aortica,239


- **Already in the database, with few missing values:** the rhythm (244 of 250), the regurgitation of the aortic, mitral and tricuspid valves (242 to 243), the valve morphology and prostheses (219 to 240), the pulmonary flow (218) and `FEVEBP` (214). For these, the extraction checks the values and fills the few gaps (e.g. the ejection fraction is written in the conclusion of 202 reports).

- **In the database, but with many missing values:** the variables of the pulmonary hypertension phenotype, `PSAP` (160), `HTP Probability` (149) and `VMaxRegTric` (99). `PSAP` is written in 122 reports, so extracting it can recover part of the missing patients. `HTP Probability` is not written in any report (no report has the word "probabilidade"). The reports only describe the morphology of the pulmonary flow (`Fluxo Pulmonar`: `Normal`, `Borderline para HTP` or `Sugestivo de HTP`), which is an indirect sign of pulmonary hypertension, not its probability. So the probability can only come from the database, and the pulmonary flow is extracted as a separate variable.

- **Only in the reports:** the remodelling, measured in most reports (septum 231, LV diameter 230, LA diameter 225, posterior wall 203, LV mass index 193, LA volume 142) and described in the conclusion (dilatation of any chamber 168, wall motion 153, hypertrophy 114), and the global function (systolic function of the LV or RV 235, pericardial effusion 231, inferior vena cava 231, TAPSE 191, S' tricuspid 176). These are the variables the extraction adds, and they supervise mainly the ECG and the fused representation.

- **Rare, but kept:** the stenosis (28 reports) and the valve area (48), because a stenosis causes a murmur and is one of the findings the PCG should reflect.

- **Too rare to be useful:** the hemodynamic data (`mPAP`, `pCWP`, `RVP`, 15 to 16 patients) and the heart rate (52 reports).

So the final prompt asks for:

- **Valves (PCG):** regurgitation, stenosis and type (native or prosthesis) of each valve, transvalvular gradients and valve area, pulmonary flow and PSAP (the pulmonary hypertension probability comes from the database);

- **Rhythm and remodelling (ECG):** rhythm, septum and posterior wall thickness, LV diameter, LA diameter and volume, LV mass index, hypertrophy, dilatation of each chamber, wall motion;

- **Global function (PCG + ECG):** LV ejection fraction and systolic function, TAPSE, S' tricuspid, pericardial effusion, inferior vena cava.